# 03c — Case-Control Matching (Problem C)

Match NTSB incidents (cases) to BTS normal flights (controls).  
This creates the labeled dataset for training the pre-flight risk model.

**Design:** 1:20 case-control ratio  
**Output:** `data/processed/preflight_casecontrol.parquet`

In [9]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

Working directory: e:\OSFDA


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NTSB_PATH = Path('data/raw/ntsb_accidents.parquet')
BTS_DIR   = Path('data/raw/bts_flights')
OUT_DIR   = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_RATIO = 20  # controls per case
SEED = 42

print(f'Case-control ratio: 1:{CONTROL_RATIO}')

Case-control ratio: 1:20


## 1. Load NTSB Cases

In [11]:
ntsb = pd.read_parquet(NTSB_PATH)
print(f'NTSB events: {len(ntsb):,}')

# Ensure we have date and airport
ntsb['event_date'] = pd.to_datetime(ntsb['event_date'], errors='coerce')

# Standardize airport codes — NTSB uses FAA LIDs, BTS uses IATA 3-letter
# Many small airports have FAA LIDs that differ from IATA
# For major airports, FAA LID == IATA code (e.g., ATL, ORD, LAX)
# Updated detection logic in Cell 4
airport_col = next((c for c in ntsb.columns if any(k in c.lower() for k in ['arpt', 'airport', 'apt_id'])), None)

if airport_col:
    print(f"Found airport column: {airport_col}")
    ntsb['airport_code'] = ntsb[airport_col].str.strip().str.upper()
    
    # Normalize: Strip leading 'K' from 4-letter US codes to match BTS 3-letter codes (e.g., KORD -> ORD)
    ntsb['airport_code'] = ntsb['airport_code'].str.replace(r'^K([A-Z]{3})$', r'\1', regex=True)
    
    has_airport = ntsb['airport_code'].notna() & (ntsb['airport_code'] != '')
    print(f'Events with airport code: {has_airport.sum():,} ({has_airport.mean()*100:.1f}%)')
    ntsb_matched = ntsb[has_airport].copy()
else:
    print('WARNING: No airport column found. Check NTSB schema.')
    ntsb_matched = ntsb.copy()

    
print(f'\nMatchable NTSB events: {len(ntsb_matched):,}')

NTSB events: 4,704
Found airport column: ev_nr_apt_id
Events with airport code: 2,821 (60.0%)

Matchable NTSB events: 2,821


## 2. Load BTS Flights

In [12]:
# Load BTS yearly parquets with downcasting to save memory
bts_files = sorted(BTS_DIR.glob('bts_20??.parquet'))
print(f'BTS files found: {[f.name for f in bts_files]}')

bts_frames = []
for f in bts_files:
    df = pd.read_parquet(f)
    
    # Downcast numeric columns to save space
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
        
    # Convert high-cardinality strings to categories
    for col in ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'ORIGIN_STATE_ABR', 'DEST_STATE_ABR']:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    print(f'  {f.name}: {len(df):,} flights')
    bts_frames.append(df)

bts = pd.concat(bts_frames, ignore_index=True)
bts['FL_DATE'] = pd.to_datetime(bts['FL_DATE'], errors='coerce')
print(f'\nTotal BTS flights: {len(bts):,}')


BTS files found: ['bts_2018.parquet', 'bts_2019.parquet', 'bts_2020.parquet']
  bts_2018.parquet: 7,206,195 flights
  bts_2019.parquet: 7,422,037 flights
  bts_2020.parquet: 4,688,354 flights

Total BTS flights: 19,316,586


## 3. Match NTSB Cases to BTS Flights

We match on **date + origin airport**. A match means an NTSB incident
occurred at the same airport on the same day as a scheduled BTS flight.

In [13]:
# 1. Create a matching key in both datasets
ntsb_matched['match_date'] = ntsb_matched['event_date'].dt.date
bts['match_date'] = bts['FL_DATE'].dt.date

# 2. Create a DataFrame of unique NTSB incident (date, airport) pairs
incident_keys_df = ntsb_matched[['match_date', 'airport_code']].dropna().copy()
incident_keys_df = incident_keys_df.drop_duplicates()
incident_keys_df['is_incident_day'] = True

# 3. Merge with BTS data to flag incident flights
bts = bts.merge(
    incident_keys_df,
    left_on=['match_date', 'ORIGIN'],
    right_on=['match_date', 'airport_code'],
    how='left'
)
bts['is_incident_day'] = bts['is_incident_day'].fillna(False)

# 4. Extract the incident flights
incident_flights = bts[bts['is_incident_day']].copy()
print(f"Matched {len(incident_flights):,} BTS flights to NTSB incidents.")


C:\Users\yashn\AppData\Local\Temp\ipykernel_10616\1422572920.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bts['is_incident_day'] = bts['is_incident_day'].fillna(False)


Matched 35,303 BTS flights to NTSB incidents.


## 4. Construct Case-Control Dataset

In [14]:
# 1. Prepare Cases (Positive samples)
cases = incident_flights.copy()
cases['incident'] = 1
n_cases = len(cases)
print(f'Cases: {n_cases:,}')

# 2. Setup Sampling parameters
n_controls_needed = n_cases * CONTROL_RATIO
print(f'Controls needed: {n_controls_needed:,} (1:{CONTROL_RATIO} ratio)')

# 3. Add temporal markers for stratified sampling
# We do this on the main bts dataframe and cases separately
# .dt.to_period('M') is efficient
bts['year_month'] = bts['FL_DATE'].dt.to_period('M')
cases['year_month'] = cases['FL_DATE'].dt.to_period('M')

# Define the pool of potential controls as a VIEW (no copy)
control_pool = bts[~bts['is_incident_day']]

# 4. Perform Stratified Sampling
case_month_dist = cases['year_month'].value_counts(normalize=True)
control_samples = []

for ym, frac in case_month_dist.items():
    n_sample = int(n_controls_needed * frac) + 1
    
    # Filter the pool for the specific month (this creates a temporary small copy)
    pool_subset = control_pool[control_pool['year_month'] == ym]
    
    n_actual = min(n_sample, len(pool_subset))
    if n_actual > 0:
        sampled = pool_subset.sample(n=n_actual, random_state=SEED)
        control_samples.append(sampled)

# 5. Finalize Controls
controls = pd.concat(control_samples, ignore_index=True)
controls['incident'] = 0
print(f'Controls sampled: {len(controls):,}')

# 6. Combine and Cleanup
# We can now create the final small dataset and delete the massive bts dataframe
dataset = pd.concat([cases, controls], ignore_index=True)
dataset = dataset.drop(columns=['is_incident_day', 'match_date', 'year_month'], errors='ignore')

# Shuffle the final small dataset
dataset = dataset.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'\nFinal dataset: {len(dataset):,} rows')
print(f'Incident rate: {dataset["incident"].mean()*100:.2f}%')

# OPTIONAL: Free up RAM immediately after this step
# del bts 


Cases: 35,303
Controls needed: 706,060 (1:20 ratio)
Controls sampled: 706,091

Final dataset: 741,394 rows
Incident rate: 4.76%


## 5. Validate

In [15]:
print('Label distribution:')
print(dataset['incident'].value_counts())

print(f'\nTemporal range: {dataset["FL_DATE"].min()} to {dataset["FL_DATE"].max()}')
print(f'Unique origins: {dataset["ORIGIN"].nunique()}')
print(f'Unique carriers: {dataset["OP_UNIQUE_CARRIER"].nunique()}')

# Sanity: cases should span the full date range
case_dates = dataset[dataset['incident'] == 1]['FL_DATE']
print(f'\nCase date range: {case_dates.min()} to {case_dates.max()}')

Label distribution:
incident
0    706091
1     35303
Name: count, dtype: int64

Temporal range: 2018-01-01 00:00:00 to 2020-12-31 00:00:00
Unique origins: 370
Unique carriers: 18

Case date range: 2018-01-11 00:00:00 to 2020-12-18 00:00:00


In [16]:
# Save
out_path = OUT_DIR / 'preflight_casecontrol.parquet'
dataset.to_parquet(out_path, index=False)
print(f'Saved {len(dataset):,} rows to {out_path}')
print(f'File size: {out_path.stat().st_size / 1e6:.1f} MB')

Saved 741,394 rows to data\processed\preflight_casecontrol.parquet
File size: 13.6 MB
